# 12.11 把聲音轉成文字後，哪些問題仍答不了？


兩段聲音都說「你好」，一段低音、一段高音。若問題是「說了什麼」，同樣轉寫「你好」可以回答；若問題是「哪段音高較高」，這兩個相同字串就不足以區分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [
    {"transcript": "你好", "pitch": "low"},
    {"transcript": "你好", "pitch": "high"},
]
print("轉寫相同", examples[0]["transcript"] == examples[1]["transcript"])
print("音高相同", examples[0]["pitch"] == examples[1]["pitch"])
for row in examples:
    print("只看轉寫", row["transcript"], "實際標籤", row["pitch"])

這是手寫的兩筆條件，印出轉寫相同 True、音高相同 False，沒有執行聽寫模型。只收到同一字串的確定性回答規則，不能根據字串本身，穩定回出兩個不同音高標籤。

自動語音辨識（Automatic Speech Recognition，ASR）主要把語音轉成文字。文字入口適合「使用者說的內容」需要文字操作的任務；純文字轉寫通常不完整保留語氣、音高、先後事件與非語音聲響。若轉寫另帶時間戳、說話者或音高資料，輸入條件已經不同，不能再說它只用了相同兩個字。

直接聲音特徵能提供這些未轉成字的線索，但模型仍要用相應材料學會判斷。多接一條入口不保證語氣辨識；也不表示所有任務都應繞過 ASR。若只需把短句記成文字，先聽寫再交文字核心是清楚的分工。

練習將問題改成「請照抄這段話」，兩筆相同轉寫是否足夠？再改成「前半段比後半段高嗎」，列出純文字遺失的條件。下一節把圖與聲音一起用時，同樣先問答案到底需要哪條線索。

<details>
<summary>回顧與查證</summary>

可回顧：[12.1頻率與振幅](https://birdhackor.github.io/tiny-perceptron-vlm/12.1.html)、[12.8聲音特徵](https://birdhackor.github.io/tiny-perceptron-vlm/12.8.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/audio.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
